# 🤖 Machine Learning — Ejercicios con Soluciones

Este notebook cubre dos algoritmos clave de ML: **SVM para clasificación** y **XGBoost para regresión**.
El flujo de trabajo es siempre el mismo en scikit-learn, lo que hace que aprender un modelo te ayude a entender todos los demás.

**Librerías necesarias:**
```bash
pip install scikit-learn xgboost matplotlib seaborn numpy pandas
```

**Cómo usar este notebook:**
1. Lee el **enunciado** y la **explicación conceptual**.
2. Intentá resolver en la celda `# TU CÓDIGO AQUÍ`.
3. Si necesitás ayuda, desplegá la **💡 Pista**.
4. Comparate con la **✅ Solución**.

---

### 🔄 El flujo universal de scikit-learn

```
1. DATOS       → Cargar y explorar
2. PREPROCESAR → Escalar, codificar, limpiar
3. DIVIDIR     → train_test_split()
4. MODELO      → modelo = Algoritmo(); modelo.fit(X_train, y_train)
5. PREDECIR    → y_pred = modelo.predict(X_test)
6. EVALUAR     → accuracy, MSE, R², confusion matrix...
7. MEJORAR     → Tuneo de hiperparámetros, cross-validation
```

In [ ]:
# 🔧 CELDA DE SETUP — Ejecutá esto primero
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Librerías base cargadas.')

# Verificar XGBoost
try:
    import xgboost as xgb
    print(f'✅ XGBoost {xgb.__version__} disponible.')
except ImportError:
    print('⚠️  XGBoost no instalado. Ejecutá: pip install xgboost')

---
## 📦 Ejercicio 1 — Clasificación con SVM (Support Vector Machine)

**Enunciado:** Cargá el dataset Iris, entrenás un modelo SVM para clasificar las especies de flores y evaluá su precisión en un conjunto de prueba.

### 📖 Explicación conceptual

**¿Qué es SVM?**

SVM busca el **hiperplano** (línea en 2D, plano en 3D, etc.) que mejor separa las clases, maximizando el margen entre los puntos más cercanos de cada clase (llamados **vectores de soporte**).

```
  ○ ○ ○         ← Clase A
         ║
    ● ● ●       ← Clase B
         ↑
    Hiperplano de separación
    (margen máximo)
```

**Parámetros clave de SVM:**

| Parámetro | Qué controla | Valores típicos |
|-----------|-------------|----------------|
| `C` | Penalización por errores (regularización). Alto C = menos errores pero puede sobreajustar | 0.1, 1, 10, 100 |
| `kernel` | Cómo transforma el espacio de features | `'linear'`, `'rbf'`, `'poly'` |
| `gamma` | Influencia de cada punto (solo en `'rbf'`) | `'scale'`, `'auto'`, 0.1, 0.01 |

**⚠️ Importante:** SVM es sensible a la escala de los datos. Siempre escalar con `StandardScaler` antes de entrenar.

**Métricas de clasificación:**

| Métrica | Fórmula | Cuándo usarla |
|---------|---------|---------------|
| Accuracy | Correctos / Total | Clases balanceadas |
| Precision | VP / (VP + FP) | Cuando los falsos positivos son costosos |
| Recall | VP / (VP + FN) | Cuando los falsos negativos son costosos |
| F1-Score | 2 × (Prec × Recall) / (Prec + Recall) | Compromiso entre ambas |

*(VP = verdaderos positivos, FP = falsos positivos, FN = falsos negativos)*

In [ ]:
# 🔧 EXPLORAMOS EL DATASET IRIS
from sklearn.datasets import load_iris

iris = load_iris()

# Convertimos a DataFrame para explorarlo mejor
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['especie'] = iris.target
df_iris['especie_nombre'] = df_iris['especie'].map(
    {0: 'setosa', 1: 'versicolor', 2: 'virginica'}
)

print(f'Shape del dataset: {df_iris.shape}')
print(f'\nDistribución de clases:')
print(df_iris['especie_nombre'].value_counts())
print(f'\nEstadísticas descriptivas:')
print(df_iris.describe().round(2))
df_iris.head()

In [ ]:
# 🔧 VISUALIZAMOS EL DATASET
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter de las dos primeras features
colores = {'setosa': 'steelblue', 'versicolor': 'tomato', 'virginica': 'seagreen'}
for especie, grupo in df_iris.groupby('especie_nombre'):
    axes[0].scatter(
        grupo['sepal length (cm)'], grupo['sepal width (cm)'],
        label=especie, alpha=0.7, s=60, color=colores[especie]
    )
axes[0].set_title('Iris: largo vs ancho del sépalo', fontweight='bold')
axes[0].set_xlabel('Largo del sépalo (cm)')
axes[0].set_ylabel('Ancho del sépalo (cm)')
axes[0].legend()

# Distribución de petal length por especie
sns.boxplot(data=df_iris, x='especie_nombre', y='petal length (cm)',
            palette=colores, ax=axes[1])
axes[1].set_title('Distribución del largo del pétalo por especie', fontweight='bold')
axes[1].set_xlabel('Especie')
axes[1].set_ylabel('Largo del pétalo (cm)')

plt.tight_layout()
plt.show()

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Importá SVC, StandardScaler, train_test_split
# 2. Separás X (features) e y (target)
# 3. Escalás con StandardScaler
# 4. Dividís en train/test (75/25)
# 5. Entrenás SVC(kernel='rbf', C=1)
# 6. Calculás accuracy, imprimís el classification_report
# 7. Graficás la matriz de confusión


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```python
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X, y = iris.data, iris.target

# Escalar ANTES de dividir evita data leakage
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)  # fit solo en train idealmente

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

modelo = SVC(kernel='rbf', C=1, random_state=42)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
```
⚠️ La forma correcta es hacer `fit` del scaler **solo en el train set** para evitar **data leakage**.

</details>

In [ ]:
# ✅ SOLUCIÓN — Parte A: SVM básico
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

X, y = iris.data, iris.target
nombres_clases = iris.target_names

# División train/test con stratify (mantiene proporción de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Pipeline: encadena el escalado y el modelo
# Ventaja: el scaler se ajusta SOLO en train, evitando data leakage
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=1, gamma='scale', random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

# Métricas
acc = accuracy_score(y_test, y_pred)
print(f'✅ Accuracy en test: {acc:.4f} ({acc*100:.1f}%)')
print(f'\n📋 Reporte por clase:')
print(classification_report(y_test, y_pred, target_names=nombres_clases))

# Cross-validation (más confiable que un solo split)
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f'🔄 Cross-validation (5 folds):')
print(f'   Scores: {cv_scores.round(3)}')
print(f'   Media:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# ✅ SOLUCIÓN — Parte B: Visualizaciones

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
    xticklabels=nombres_clases, yticklabels=nombres_clases,
    linewidths=0.5, cbar=False
)
axes[0].set_title('Matriz de confusión\nSVM (kernel=rbf)', fontweight='bold')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicho')

# 2. Comparación de kernels
kernels  = ['linear', 'rbf', 'poly', 'sigmoid']
acc_kern = []
for k in kernels:
    p = Pipeline([('sc', StandardScaler()), ('svm', SVC(kernel=k, random_state=42))])
    score = cross_val_score(p, X, y, cv=5).mean()
    acc_kern.append(score)

bars = axes[1].bar(kernels, acc_kern,
                   color=sns.color_palette('muted', len(kernels)), edgecolor='white')
axes[1].set_title('Accuracy por kernel (CV=5)', fontweight='bold')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.8, 1.02)
for bar, val in zip(bars, acc_kern):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

# 3. Efecto del parámetro C
C_valores = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
acc_train, acc_test_c = [], []
for C in C_valores:
    p = Pipeline([('sc', StandardScaler()), ('svm', SVC(kernel='rbf', C=C, random_state=42))])
    p.fit(X_train, y_train)
    acc_train.append(accuracy_score(y_train, p.predict(X_train)))
    acc_test_c.append(accuracy_score(y_test,  p.predict(X_test)))

axes[2].semilogx(C_valores, acc_train, 'o-', color='steelblue', label='Train', linewidth=2)
axes[2].semilogx(C_valores, acc_test_c, 's--', color='tomato',    label='Test',  linewidth=2)
axes[2].set_title('Efecto del parámetro C\n(underfitting ← → overfitting)', fontweight='bold')
axes[2].set_xlabel('C (escala logarítmica)')
axes[2].set_ylabel('Accuracy')
axes[2].legend()
axes[2].set_ylim(0.9, 1.01)

plt.suptitle('📊 Análisis completo del modelo SVM — Iris', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ✅ SOLUCIÓN — Parte C: Búsqueda de hiperparámetros con GridSearchCV
from sklearn.model_selection import GridSearchCV

# Definimos la grilla de hiperparámetros a probar
param_grid = {
    'svm__C':      [0.1, 1, 10, 100],
    'svm__kernel': ['rbf', 'linear', 'poly'],
    'svm__gamma':  ['scale', 'auto']
}

pipeline_grid = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(random_state=42))
])

# GridSearchCV prueba todas las combinaciones con cross-validation
grid_search = GridSearchCV(
    pipeline_grid,
    param_grid,
    cv=5,              # 5-fold cross validation
    scoring='accuracy',
    n_jobs=-1,         # Usa todos los cores disponibles
    verbose=0
)
grid_search.fit(X_train, y_train)

print('🔍 GridSearchCV — Mejores hiperparámetros encontrados:')
for param, valor in grid_search.best_params_.items():
    print(f'   {param}: {valor}')
print(f'\n   Mejor CV accuracy: {grid_search.best_score_:.4f}')

# Evaluamos el mejor modelo en test
mejor_modelo = grid_search.best_estimator_
y_pred_best  = mejor_modelo.predict(X_test)
acc_best     = accuracy_score(y_test, y_pred_best)
print(f'   Accuracy en test (mejor modelo): {acc_best:.4f}')

# Top 5 combinaciones
resultados = pd.DataFrame(grid_search.cv_results_)
cols_mostrar = ['param_svm__C', 'param_svm__kernel', 'mean_test_score', 'std_test_score']
print('\n📊 Top 5 combinaciones:')
print(
    resultados[cols_mostrar]
    .sort_values('mean_test_score', ascending=False)
    .head(5)
    .rename(columns={
        'param_svm__C': 'C',
        'param_svm__kernel': 'kernel',
        'mean_test_score': 'accuracy_media',
        'std_test_score': 'std'
    })
    .round(4)
    .to_string(index=False)
)

### 🔍 Conceptos importantes que aprendiste en este ejercicio

| Concepto | Descripción |
|----------|-------------|
| **Pipeline** | Encadena pasos (scaler → modelo) para evitar data leakage y simplificar código |
| **Data leakage** | Error de ajustar el scaler con datos de test, haciendo que el modelo "trampe" |
| **Cross-validation** | Divide los datos en K partes, entrena K veces, promedia. Más confiable que un solo split |
| **GridSearchCV** | Prueba todas las combinaciones de hiperparámetros y elige la mejor mediante CV |
| **Underfitting/Overfitting** | Muy poco C → underfitting (modelo demasiado simple). Muy alto C → overfitting (memoriza el train) |

---

## 📦 Ejercicio 2 — Regresión con XGBoost

**Enunciado:** Usá XGBoost para predecir precios de viviendas. Evaluá con MSE y analizá la importancia de las features.

### 📖 Explicación conceptual

**¿Qué es XGBoost?**

XGBoost (eXtreme Gradient Boosting) es uno de los algoritmos más potentes para datos tabulares. Se basa en **árboles de decisión** combinados con **boosting**:

```
Árbol 1: predicción inicial (puede ser mala)
   ↓ errores residuales
Árbol 2: aprende de los errores del árbol 1
   ↓ errores residuales
Árbol 3: aprende de los errores del árbol 2
   ...
Árbol N: refinamiento final
─────────────────────────────
Predicción final = suma ponderada de todos los árboles
```

**Parámetros clave de XGBoost:**

| Parámetro | Qué controla | Rango típico |
|-----------|-------------|-------------|
| `n_estimators` | Número de árboles | 100–1000 |
| `max_depth` | Profundidad máxima de cada árbol | 3–8 |
| `learning_rate` | Cuánto aprende cada árbol (η) | 0.01–0.3 |
| `subsample` | % de datos usados por árbol | 0.6–1.0 |
| `colsample_bytree` | % de features por árbol | 0.6–1.0 |

**Métricas de regresión:**

| Métrica | Fórmula | Unidades |
|---------|---------|----------|
| MAE | promedio de \|real - pred\| | mismas que y |
| MSE | promedio de (real - pred)² | y² |
| RMSE | √MSE | mismas que y |
| R² | 1 - MSE/varianza_y | 0 a 1 (1 = perfecto) |

In [ ]:
# 🔧 DATASET: California Housing (más realista que Iris para regresión)
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

df_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
df_housing['precio'] = housing.target  # precio en $100,000

print(f'Dataset California Housing:')
print(f'  Shape: {df_housing.shape}')
print(f'  Features: {housing.feature_names}')
print(f'  Target: precio de vivienda (en $100,000)')
print(f'\nDescripción del target:')
print(df_housing['precio'].describe().round(3))
print(f'\nValores nulos: {df_housing.isnull().sum().sum()}')
df_housing.head()

In [ ]:
# 🔧 EXPLORACIÓN VISUAL DEL DATASET
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(housing.feature_names):
    axes[i].scatter(
        df_housing[col], df_housing['precio'],
        alpha=0.1, s=5, color='steelblue'
    )
    axes[i].set_xlabel(col, fontsize=9)
    axes[i].set_ylabel('Precio' if i % 4 == 0 else '')
    r = df_housing[col].corr(df_housing['precio'])
    axes[i].set_title(f'r = {r:.3f}', fontsize=10)

plt.suptitle('Relación de cada feature con el precio de vivienda', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n🔗 Correlaciones con el precio (ordenadas):')
corrs = df_housing.corr()['precio'].drop('precio').sort_values(key=abs, ascending=False)
print(corrs.round(3))

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Separás X e y del df_housing
# 2. Dividís en train (80%) y test (20%)
# 3. Entrenás XGBRegressor con parámetros básicos
# 4. Calculás MAE, RMSE y R² en el test set
# 5. Graficás: valores reales vs predichos
# 6. Mostrás la importancia de cada feature


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```python
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X = df_housing.drop('precio', axis=1)
y = df_housing['precio']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo_xgb = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    verbosity=0
)
modelo_xgb.fit(X_train, y_train)
y_pred = modelo_xgb.predict(X_test)
```
Para la importancia: `modelo_xgb.feature_importances_` devuelve un array con el peso de cada feature.

</details>

In [ ]:
# ✅ SOLUCIÓN — Parte A: XGBoost básico
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

X = df_housing.drop('precio', axis=1)
y = df_housing['precio']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {len(X_train):,} muestras | Test: {len(X_test):,} muestras')

# Entrenamos XGBoost
modelo_xgb = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
modelo_xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],   # Para monitorear overfitting
    verbose=False
)

y_pred_xgb = modelo_xgb.predict(X_test)

# Métricas
def metricas(nombre, y_real, y_pred):
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2   = r2_score(y_real, y_pred)
    print(f'{nombre}:')
    print(f'   MAE:  {mae:.4f}  (error promedio: ${mae*100_000:,.0f})')
    print(f'   RMSE: {rmse:.4f} (error típico:   ${rmse*100_000:,.0f})')
    print(f'   R²:   {r2:.4f}  (explica el {r2*100:.1f}% de la variación)')
    return mae, rmse, r2

print('\n📊 Métricas en el conjunto de TEST:')
mae_x, rmse_x, r2_x = metricas('XGBoost', y_test, y_pred_xgb)

# Comparamos con modelos más simples
print()
lr = LinearRegression()
lr.fit(X_train, y_train)
mae_lr, rmse_lr, r2_lr = metricas('Regresión Lineal (baseline)', y_test, lr.predict(X_test))

print()
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
mae_rf, rmse_rf, r2_rf = metricas('Random Forest', y_test, rf.predict(X_test))

In [ ]:
# ✅ SOLUCIÓN — Parte B: Visualizaciones completas

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# 1. Real vs Predicho — XGBoost
axes[0,0].scatter(y_test, y_pred_xgb, alpha=0.2, s=8, color='steelblue')
lim = [y_test.min(), y_test.max()]
axes[0,0].plot(lim, lim, 'r--', lw=2, label='Predicción perfecta')
axes[0,0].set_title(f'XGBoost — Real vs Predicho\nR² = {r2_x:.3f}', fontweight='bold')
axes[0,0].set_xlabel('Precio real ($100K)')
axes[0,0].set_ylabel('Precio predicho ($100K)')
axes[0,0].legend()

# 2. Residuos — XGBoost
residuos = y_test.values - y_pred_xgb
axes[0,1].scatter(y_pred_xgb, residuos, alpha=0.2, s=8, color='steelblue')
axes[0,1].axhline(0, color='tomato', lw=2, linestyle='--')
axes[0,1].set_title('Residuos del modelo XGBoost', fontweight='bold')
axes[0,1].set_xlabel('Valor predicho')
axes[0,1].set_ylabel('Residuo (real - predicho)')

# 3. Distribución de residuos
axes[0,2].hist(residuos, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0,2].axvline(0, color='tomato', lw=2, linestyle='--')
axes[0,2].set_title('Distribución de residuos\n(ideal: centrada en 0)', fontweight='bold')
axes[0,2].set_xlabel('Residuo')
axes[0,2].set_ylabel('Frecuencia')

# 4. Importancia de features — XGBoost
importancias = pd.Series(
    modelo_xgb.feature_importances_,
    index=housing.feature_names
).sort_values(ascending=True)

colores_imp = ['tomato' if v == importancias.max() else 'steelblue' for v in importancias.values]
importancias.plot(kind='barh', ax=axes[1,0], color=colores_imp, edgecolor='white')
axes[1,0].set_title('Importancia de features\nXGBoost', fontweight='bold')
axes[1,0].set_xlabel('Importancia relativa')

# 5. Comparación de modelos
modelos  = ['Reg. Lineal', 'Random Forest', 'XGBoost']
r2_vals  = [r2_lr, r2_rf, r2_x]
rmse_vals= [rmse_lr, rmse_rf, rmse_x]
colores_mod = ['#FFC107', '#4CAF50', '#2196F3']

bars = axes[1,1].bar(modelos, r2_vals, color=colores_mod, edgecolor='white')
axes[1,1].set_title('Comparación R² por modelo', fontweight='bold')
axes[1,1].set_ylabel('R²')
axes[1,1].set_ylim(0, 1.1)
for bar, val in zip(bars, r2_vals):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                   f'{val:.3f}', ha='center', fontweight='bold')

bars2 = axes[1,2].bar(modelos, rmse_vals, color=colores_mod, edgecolor='white')
axes[1,2].set_title('Comparación RMSE por modelo\n(menor = mejor)', fontweight='bold')
axes[1,2].set_ylabel('RMSE ($100K)')
for bar, val in zip(bars2, rmse_vals):
    axes[1,2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                   f'{val:.3f}', ha='center', fontweight='bold')

plt.suptitle('📊 Análisis completo — XGBoost para precios de vivienda', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ✅ SOLUCIÓN — Parte C: Tuneo de hiperparámetros con early stopping

print('🔍 Análisis del efecto de n_estimators y learning_rate...')

configs = [
    {'n_estimators': 100,  'learning_rate': 0.3,   'label': 'lr=0.30, n=100'},
    {'n_estimators': 200,  'learning_rate': 0.1,   'label': 'lr=0.10, n=200'},
    {'n_estimators': 500,  'learning_rate': 0.05,  'label': 'lr=0.05, n=500'},
    {'n_estimators': 1000, 'learning_rate': 0.01,  'label': 'lr=0.01, n=1000'},
]

resultados_tuneo = []
for cfg in configs:
    m = xgb.XGBRegressor(
        n_estimators=cfg['n_estimators'],
        learning_rate=cfg['learning_rate'],
        max_depth=5, subsample=0.8,
        random_state=42, verbosity=0
    )
    m.fit(X_train, y_train, verbose=False)
    preds = m.predict(X_test)
    r2    = r2_score(y_test, preds)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    resultados_tuneo.append({'config': cfg['label'], 'R²': r2, 'RMSE': rmse})

df_tuneo = pd.DataFrame(resultados_tuneo).set_index('config').round(4)
print('\n📊 Resultados del tuneo:')
print(df_tuneo.to_string())

mejor_config = df_tuneo['R²'].idxmax()
print(f'\n🏆 Mejor configuración: {mejor_config} (R² = {df_tuneo.loc[mejor_config, "R²"]:.4f})')

print('\n💡 Conclusión:')
print('   Learning rate bajo + más árboles suele dar mejores resultados,')
print('   pero aumenta el tiempo de entrenamiento. Hay que balancear.')

### 🔍 Conceptos importantes que aprendiste en este ejercicio

| Concepto | Descripción |
|----------|-------------|
| **Boosting** | Combina modelos débiles (árboles simples) en secuencia, cada uno corrigiendo al anterior |
| **Feature importance** | XGBoost puede decirte qué variables aportan más al modelo |
| **Residuos** | La diferencia entre valor real y predicho. Si no son aleatorios, el modelo tiene sesgos |
| **Learning rate + n_estimators** | Relación inversa: lr bajo → necesitás más árboles, pero el resultado suele ser más robusto |
| **Baseline** | Siempre comparar con un modelo simple (regresión lineal) para saber cuánto mejora el modelo complejo |

---

## 🏆 Resumen del notebook

| Ejercicio | Algoritmo | Tipo | Métricas |
|-----------|-----------|------|----------|
| 1 | SVM | Clasificación | Accuracy, Precision, Recall, F1, Matriz de confusión |
| 2 | XGBoost | Regresión | MAE, RMSE, R², Feature importance |

### 🔑 Reglas de oro del ML

1. **Siempre explorá los datos** antes de entrenar (EDA).
2. **Siempre escalá** los datos para algoritmos basados en distancias (SVM, KNN).
3. **Usá Pipeline** para evitar data leakage.
4. **Validá con cross-validation**, no solo con un split.
5. **Siempre compará con un baseline** (modelo simple).
6. **Interpretá los residuos** para detectar patrones no capturados.

### 💪 Desafíos extra
1. **Ej. 1:** Cargá el dataset `breast_cancer` (cancer de mama) y aplicá SVM. ¿Qué métrica importa más aquí: precision o recall? ¿Por qué?
2. **Ej. 1:** Implementá `RandomizedSearchCV` en vez de `GridSearchCV`. ¿Cuánto más rápido es?
3. **Ej. 2:** Agregá regularización L1 (`reg_alpha`) y L2 (`reg_lambda`) al XGBoost y medí el impacto.
4. **Ej. 2:** Usá `SHAP` (`pip install shap`) para interpretar por qué el modelo toma cada predicción.
5. **Ambos:** Probá `LightGBM` (alternativa a XGBoost) y comparalos en velocidad y accuracy.